In [4]:
import os
import json
import pandas as pd

In [5]:
ROOT_DIR = r"runs\optuna\encoder"

results = []

for folder in os.listdir(ROOT_DIR):
    if not folder.startswith("trial_"):
        continue

    trial_dir = os.path.join(ROOT_DIR, folder)

    if not os.path.isdir(trial_dir):
        continue

    trial_num = int(folder.split("_")[-1])

    metrics_path = os.path.join(trial_dir, "metrics.csv")
    config_path = os.path.join(trial_dir, "training_config.json")

    if not os.path.exists(metrics_path):
        print(f"Skipping trial {trial_num}: metrics.csv not found")
        continue

    if not os.path.exists(config_path):
        print(f"Skipping trial {trial_num}: training_config.json not found")
        continue

    # Read metrics
    metrics_df = pd.read_csv(metrics_path)

    if "val/mAP_50" not in metrics_df.columns:
        print(f"Skipping trial {trial_num}: val/mAP_50 not found")
        continue

    best_map50 = metrics_df["val/mAP_50"].max()

    # Read training config
    with open(config_path, "r") as f:
        config = json.load(f)["train_config"]

    results.append({
        "trial": trial_num,
        "lr": config.get("lr"),
        "lr_encoder": config.get("lr_encoder"),
        "weight_decay": config.get("weight_decay"),
        "best_mAP50": best_map50,
    })


results_df = pd.DataFrame(results)

# Best mAP50 first
results_df = results_df.sort_values(
    "best_mAP50",
    ascending=False
).reset_index(drop=True)


# Format values before saving to CSV
def fmt(x):
    if pd.isna(x):
        return ""
    if abs(x) < 0.01:
        return f"{x:.2e}"
    return f"{x:.2f}"


for col in [
    "lr",
    "lr_encoder",
    "weight_decay",
    "best_mAP50",
]:
    results_df[col] = results_df[col].apply(fmt)

results_df["trial"] = results_df["trial"].astype(int)

display(results_df)

,trial,lr,lr_encoder,weight_decay,best_mAP50
0,1,6.96e-05,2.43e-04,1.55e-05,0.72
1,4,8.80e-05,1.96e-04,2.07e-04,0.71
2,8,1.54e-04,9.21e-05,4.03e-04,0.70
3,5,1.27e-04,7.72e-05,2.94e-05,0.70
4,7,5.38e-05,2.13e-04,1.72e-05,0.69
5,0,2.99e-04,5.73e-05,7.00e-05,0.69
6,2,5.56e-05,6.68e-05,2.99e-04,0.68
7,6,2.54e-04,3.63e-05,7.81e-05,0.67
8,3,1.29e-04,6.31e-05,5.50e-04,0.67
9,9,3.48e-05,2.33e-04,4.07e-05,0.30


In [6]:
results_df.to_csv("encoder tuning.csv")

In [7]:
ROOT_DIR = r"runs\optuna\augmentation"

results = []

for folder in os.listdir(ROOT_DIR):
    if not folder.startswith("trial_"):
        continue

    trial_dir = os.path.join(ROOT_DIR, folder)

    if not os.path.isdir(trial_dir):
        continue

    trial_num = int(folder.split("_")[-1])

    metrics_path = os.path.join(trial_dir, "metrics.csv")
    config_path = os.path.join(trial_dir, "training_config.json")

    if not os.path.exists(metrics_path):
        print(f"Skipping trial {trial_num}: metrics.csv not found")
        continue

    if not os.path.exists(config_path):
        print(f"Skipping trial {trial_num}: training_config.json not found")
        continue

    # Read metrics
    metrics_df = pd.read_csv(metrics_path)

    if "val/mAP_50" not in metrics_df.columns:
        print(f"Skipping trial {trial_num}: val/mAP_50 not found")
        continue

    best_map50 = metrics_df["val/mAP_50"].max()

    # Read augmentation config
    with open(config_path, "r") as f:
        aug_config = json.load(f)["train_config"]["aug_config"]

    results.append({
        "trial": trial_num,
        "flip_p": aug_config["HorizontalFlip"]["p"],
        "rotate_limit": aug_config["Rotate"]["limit"],
        "rotate_p": aug_config["Rotate"]["p"],
        "brightness_p": aug_config["RandomBrightnessContrast"]["p"],
        "blur_p": aug_config["GaussianBlur"]["p"],
        "best_mAP50": best_map50,
    })


results_df = pd.DataFrame(results)

# Best mAP50 first
results_df = results_df.sort_values(
    "best_mAP50",
    ascending=False
).reset_index(drop=True)


# Format values before saving to CSV
def fmt(x):
    if pd.isna(x):
        return ""
    if abs(x) < 0.01:
        return f"{x:.2e}"
    return f"{x:.2f}"


for col in [
    "flip_p",
    "rotate_p",
    "brightness_p",
    "blur_p",
    "best_mAP50",
]:
    results_df[col] = results_df[col].apply(fmt)

results_df["trial"] = results_df["trial"].astype(int)
results_df["rotate_limit"] = results_df["rotate_limit"].astype(int)

display(results_df)

,trial,flip_p,rotate_limit,rotate_p,brightness_p,blur_p,best_mAP50
0,4,0.51,1,0.12,0.18,0.09,0.73
1,0,0.39,12,0.29,0.30,0.03,0.72
2,3,0.29,3,0.21,0.22,0.06,0.72
3,5,0.59,2,0.21,0.30,9.29e-03,0.72
4,6,0.50,2,0.03,0.47,0.19,0.72
5,7,0.60,3,0.04,0.34,0.09,0.72
6,1,0.28,0,0.35,0.30,0.14,0.72
7,2,0.21,12,0.33,0.11,0.04,0.72


In [8]:
results_df.to_csv("augmentation tuning.csv")